In [2]:
import pandas as pd
import numpy as np

TWEET_DATA = pd.read_csv("mbg-2023-2026-clean-duplicate-manual.csv", encoding='latin1')

TWEET_DATA.head()

,No,full_text,username
0,1,"Jelang Lebaran, Perusahaan Mulai Booking Bus u...",InfoMakan2
1,2,@anggapph2 anggaran segitu aja masih banyak se...,dranux
2,3,@sfMarinaH keren sih ada program Makan Siang G...,nurmaphie
3,4,Prabowo menunjukkan komitmen nyata pada masa d...,rismamelatiku
4,5,"Program Prabowo: makan gratis kpd pelajar, ibu...",narkosun


In [3]:
# ------ Case Folding --------
# gunakan fungsi Series.str.lower() pada Pandas
TWEET_DATA['full_text'] = TWEET_DATA['full_text'].str.lower()


print('Case Folding Result : \n')
print(TWEET_DATA['full_text'].head(5))
print('\n\n\n')

Case Folding Result : 

0    jelang lebaran, perusahaan mulai booking bus u...
1    @anggapph2 anggaran segitu aja masih banyak se...
2    @sfmarinah keren sih ada program makan siang g...
3    prabowo menunjukkan komitmen nyata pada masa d...
4    program prabowo: makan gratis kpd pelajar, ibu...
Name: full_text, dtype: object






In [4]:
import string
import re #regex library

# import word_tokenize & FreqDist from NLTK
from nltk.tokenize import word_tokenize
from nltk.probability import FreqDist
import nltk
nltk.download('punkt_tab')

# ------ Tokenizing ---------

def remove_tweet_special(text):
    # remove tab, new line, ans back slice
    text = text.replace('\t'," ").replace('\n'," ").replace('\\u'," ").replace('\\',"")
    # remove non ASCII (emoticon, chinese word, .etc)
    text = text.encode('ascii', 'replace').decode('ascii')
    # remove mention, link, hashtag
    text = ' '.join(re.sub("([@#][A-Za-z0-9]+)|(\w+:\/\/\S+)"," ", text).split())
    # remove incomplete URL
    return text.replace("http://", " ").replace("https://", " ")

TWEET_DATA['full_text'] = TWEET_DATA['full_text'].apply(remove_tweet_special)

#remove number
def remove_number(text):
    return  re.sub(r"\d+", "", text)

TWEET_DATA['full_text'] = TWEET_DATA['full_text'].apply(remove_number)

#remove punctuation
def remove_punctuation(text):
    return text.translate(str.maketrans("","",string.punctuation))

TWEET_DATA['full_text'] = TWEET_DATA['full_text'].apply(remove_punctuation)

#remove whitespace leading & trailing
def remove_whitespace_LT(text):
    return text.strip()

TWEET_DATA['full_text'] = TWEET_DATA['full_text'].apply(remove_whitespace_LT)

#remove multiple whitespace into single whitespace
def remove_whitespace_multiple(text):
    return re.sub('\s+',' ',text)

TWEET_DATA['full_text'] = TWEET_DATA['full_text'].apply(remove_whitespace_multiple)

# remove single char
def remove_singl_char(text):
    return re.sub(r"\b[a-zA-Z]\b", "", text)

TWEET_DATA['full_text'] = TWEET_DATA['full_text'].apply(remove_singl_char)

# NLTK word rokenize
def word_tokenize_wrapper(text):
    return word_tokenize(text)

TWEET_DATA['tweet_tokens'] = TWEET_DATA['full_text'].apply(word_tokenize_wrapper)

print('Tokenizing Result : \n')
print(TWEET_DATA['tweet_tokens'].head())
print('\n\n\n')

<>:18: SyntaxWarning: invalid escape sequence '\w'
<>:44: SyntaxWarning: invalid escape sequence '\s'
<>:18: SyntaxWarning: invalid escape sequence '\w'
<>:44: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_8887/681984018.py:18: SyntaxWarning: invalid escape sequence '\w'
  text = ' '.join(re.sub("([@#][A-Za-z0-9]+)|(\w+:\/\/\S+)"," ", text).split())
/tmp/ipykernel_8887/681984018.py:44: SyntaxWarning: invalid escape sequence '\s'
  return re.sub('\s+',' ',text)
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Tokenizing Result : 

0    [jelang, lebaran, perusahaan, mulai, booking, ...
1    [anggaran, segitu, aja, masih, banyak, sekolah...
2    [keren, sih, ada, program, makan, siang, grati...
3    [prabowo, menunjukkan, komitmen, nyata, pada, ...
4    [program, prabowo, makan, gratis, kpd, pelajar...
Name: tweet_tokens, dtype: object






In [5]:
# NLTK calc frequency distribution
def freqDist_wrapper(text):
    return FreqDist(text)

TWEET_DATA['tweet_tokens_fdist'] = TWEET_DATA['tweet_tokens'].apply(freqDist_wrapper)

print('Frequency Tokens : \n')
print(TWEET_DATA['tweet_tokens_fdist'].head().apply(lambda x : x.most_common()))

Frequency Tokens : 

0    [(jelang, 1), (lebaran, 1), (perusahaan, 1), (...
1    [(program, 3), (banyak, 2), (gratis, 2), (angg...
2    [(secara, 2), (keren, 1), (sih, 1), (ada, 1), ...
3    [(prabowo, 1), (menunjukkan, 1), (komitmen, 1)...
4    [(makan, 4), (pelajar, 4), (gratis, 2), (juta,...
Name: tweet_tokens_fdist, dtype: object


In [8]:
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

# ----------------------- get stopword from NLTK stopword -------------------------------
# get stopword indonesia
list_stopwords = stopwords.words('indonesian')


# ---------------------------- manualy add stopword  ------------------------------------
# append additional stopword
list_stopwords.extend(["yg", "dg", "rt", "dgn", "ny", "d", 'klo',
                       'kalo', 'amp', 'biar', 'bikin', 'bilang',
                       'gak', 'ga', 'krn', 'nya', 'nih', 'sih',
                       'si', 'tau', 'tdk', 'tuh', 'utk', 'ya',
                       'jd', 'jgn', 'sdh', 'aja', 'n', 't',
                       'nyg', 'hehe', 'pen', 'u', 'nan', 'loh', 'rt',
                       '&amp', 'yah'])

# ----------------------- add stopword from txt file ------------------------------------
# read txt stopword using pandas
txt_stopword = pd.read_csv("stopwords-id.txt", names= ["stopwords"], header = None)

# convert stopword string to list & append additional stopword
list_stopwords.extend(txt_stopword["stopwords"][0].split(' '))

# ---------------------------------------------------------------------------------------

# convert list to dictionary
list_stopwords = set(list_stopwords)


#remove stopword pada list token
def stopwords_removal(words):
    return [word for word in words if word not in list_stopwords]

TWEET_DATA['tweet_tokens_WSW'] = TWEET_DATA['tweet_tokens'].apply(stopwords_removal)


print(TWEET_DATA['tweet_tokens_WSW'].head())

0    [jelang, lebaran, perusahaan, booking, bus, pr...
1    [anggaran, segitu, sekolah, tambahan, orang, t...
2    [keren, program, makan, siang, gratis, kayak, ...
3    [prabowo, komitmen, nyata, makan, siang, grati...
4    [program, prabowo, makan, gratis, kpd, pelajar...
Name: tweet_tokens_WSW, dtype: object


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
# normalizad_word = pd.read_csv("slang.csv")

# normalizad_word_dict = {}

# for index, row in normalizad_word.iterrows():
#     if row[0] not in normalizad_word_dict:
#         normalizad_word_dict[row[0]] = row[1]

# def normalized_term(document):
#     return [normalizad_word_dict[term] if term in normalizad_word_dict else term for term in document]

# TWEET_DATA['tweet_normalized'] = TWEET_DATA['tweet_tokens_WSW'].apply(normalized_term)

# TWEET_DATA['tweet_normalized'].head(10)

In [11]:
import ast

# 1. Load data dari CSV (Existing code)
normalizad_word = pd.read_csv("slang.csv")
normalizad_word_dict = {}

for index, row in normalizad_word.iterrows():
    if row[0] not in normalizad_word_dict:
        normalizad_word_dict[row[0]] = row[1]

# 2. Load data tambahan dari file .txt
# Ganti 'slang_tambahan.txt' dengan nama file txt Anda
with open('combined_slang_words.txt', 'r') as f:
    data_txt = f.read()
    # Menggunakan ast.literal_eval untuk mengubah string format dict menjadi dict asli
    txt_dict = ast.literal_eval(data_txt)

# 3. Gabungkan dictionary dari TXT ke dictionary utama
normalizad_word_dict.update(txt_dict)

# 4. Fungsi Preprocessing (Tetap sama)
def normalized_term(document):
    return [normalizad_word_dict[term] if term in normalizad_word_dict else term for term in document]

# 5. Eksekusi pada DataFrame
TWEET_DATA['tweet_normalized'] = TWEET_DATA['tweet_tokens_WSW'].apply(normalized_term)

# Cek hasil
print(TWEET_DATA['tweet_normalized'].head(10))

0    [jelang, lebaran, perusahaan, booking, bus, pr...
1    [anggaran, segitu, sekolah, tambahan, orang, t...
2    [keren, program, makan, siang, gratis, seperti...
3    [prabowo, komitmen, nyata, makan, siang, grati...
4    [program, prabowo, makan, gratis, kepada, pela...
5    [makanan, gratis, bergizi, ibuibu, hamil, memp...
6    [aneh, wacana, makan, gratis, tujuannya, mence...
7    [raharjo, anak, sekolah, dijanjikan, bayi, bal...
8    [ide, pembagian, susu, makan, siang, bergizi, ...
9    [program, makan, siang, gratis, anak, sarana, ...
Name: tweet_normalized, dtype: object


/tmp/ipykernel_8887/497801340.py:8: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if row[0] not in normalizad_word_dict:
/tmp/ipykernel_8887/497801340.py:9: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  normalizad_word_dict[row[0]] = row[1]


In [12]:
# Install library swifter
%pip install swifter

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 14.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for swifter: filename=swifter-1.4.0-py3-none-any.whl size=16505 sha256=d38c263a27f4e09a1b90c8c8995af956e8f2975e107cd93e62481027c85312b1
  Stored in directory: /root/.cache/pip/wheels/d9/31/ff/ff51141a088571a9f672449e5aad5ea8bb35ca5d95ba135f30
Successfully built swifter


In [13]:
# Install Sastrawi package
%pip install Sastrawi

# import Sastrawi package
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import swifter


# create stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# stemmed
def stemmed_wrapper(term):
    return stemmer.stem(term)

term_dict = {}

for document in TWEET_DATA['tweet_normalized']:
    for term in document:
        if term not in term_dict:
            term_dict[term] = ' '

print(len(term_dict))
print("------------------------")

for term in term_dict:
    term_dict[term] = stemmed_wrapper(term)
    print(term,":" ,term_dict[term])

print(term_dict)
print("------------------------")


# apply stemmed term to dataframe
def get_stemmed_term(document):
    return [term_dict[term] for term in document]

TWEET_DATA['tweet_tokens_stemmed'] = TWEET_DATA['tweet_normalized'].swifter.apply(get_stemmed_term)
print(TWEET_DATA['tweet_tokens_stemmed'])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 4.6 MB/s eta 0:00:00
2251
------------------------
jelang : jelang
lebaran : lebaran
perusahaan : usaha
booking : booking
bus : bus
program : program
mudik : mudik
gratis : gratis
anggaran : anggar
segitu : segitu
sekolah : sekolah
tambahan : tambah
orang : orang
tua : tua
makan : makan
siang : siang
mark : mark
atas : atas
saya : saya
sd : sd
snack : snack
bergizi : gizi
gabungan : gabung
pkk : pkk
keren : keren
seperti : seperti
gini : gin
pertumbuhan : tumbuh
perkembangan : kembang
anak : anak
indonesia : indonesia
fisik : fisik
kemampuan : mampu
iq : iq
mengikuti : ikut
asupan : asupan
nutrisi : nutrisi
prabowo : prabowo
komitmen : komitmen
nyata : nyata
anakanak : anakanak
kepada : kepada
pelajar : ajar
hamil : hamil
juta : juta
mari : mari
hitung : hitung
biaya : biaya
rp : rp
ribu : ribu
porsi : porsi
harga : harga
dianggap : anggap
lengkap : lengkap
dalam : dalam
negara : negara
makanan : makan
ibuibu : ibuibu
memperkua

Pandas Apply:   0%|          | 0/462 [00:00<?, ?it/s]

0      [jelang, lebaran, usaha, booking, bus, program...
1      [anggar, segitu, sekolah, tambah, orang, tua, ...
2      [keren, program, makan, siang, gratis, seperti...
3      [prabowo, komitmen, nyata, makan, siang, grati...
4      [program, prabowo, makan, gratis, kepada, ajar...
                             ...                        
457    [program, makan, gizi, gratis, gerak, roda, ek...
458    [prabowo, kamu, kasih, makan, jabat, mbg, sppg...
459    [program, makan, gizi, gratis, mbg, hadir, lan...
460    [suap, program, makan, gizi, gratis, langkah, ...
461    [jakarta, perintah, indonesia, rencana, mangka...
Name: tweet_tokens_stemmed, Length: 462, dtype: object


In [14]:
# Save ke CSV, untuk save ke CSV
TWEET_DATA.to_csv("Text_Preprocessing.csv")